In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install fastapi uvicorn httpx pydantic nest-asyncio

In [3]:
import nest_asyncio
nest_asyncio.apply()

import json
import random
import sqlite3
import uvicorn
import string
from contextlib import contextmanager
from typing import Optional
 
import httpx
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

In [4]:
DB_PATH = "flip_data.db"
app = FastAPI(title="Flip Data Engineer API", version="1.0.0")

### DATABASE

In [5]:
@contextmanager
def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()

In [6]:
def init_db():
    with get_conn() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS ability_effects (
                id                 INTEGER PRIMARY KEY AUTOINCREMENT,
                raw_id             TEXT    NOT NULL,
                user_id            TEXT    NOT NULL,
                pokemon_ability_id INTEGER NOT NULL,
                effect             TEXT,
                language           TEXT,
                short_effect       TEXT
            )
        """)

In [7]:
def save_entries(raw_id, user_id, pokemon_ability_id, effect_entries):
    rows = [
        (
            raw_id,
            user_id,
            pokemon_ability_id,
            entry.get("effect", ""),
            json.dumps(entry.get("language", {})),
            entry.get("short_effect", ""),
        )
        for entry in effect_entries
    ]
    with get_conn() as conn:
        conn.executemany("""
            INSERT INTO ability_effects
                (raw_id, user_id, pokemon_ability_id, effect, language, short_effect)
            VALUES (?, ?, ?, ?, ?, ?)
        """, rows)

In [8]:
def get_entries(raw_id, user_id, pokemon_ability_id):
    with get_conn() as conn:
        rows = conn.execute("""
            SELECT effect, language, short_effect
            FROM ability_effects
            WHERE raw_id = ? AND user_id = ? AND pokemon_ability_id = ?
        """, (raw_id, user_id, pokemon_ability_id)).fetchall()
 
    return [
        {
            "effect": r["effect"],
            "language": json.loads(r["language"]),
            "short_effect": r["short_effect"],
        }
        for r in rows
    ]

### MODELS

In [9]:
class AbilityRequest(BaseModel):
    raw_id: Optional[str] = None
    user_id: Optional[str] = None
    pokemon_ability_id: str

### HELPERS

In [10]:
def make_raw_id() -> str:
    """Random alphanumeric string, exactly 13 characters."""
    return "".join(random.choices(string.ascii_lowercase + string.digits, k=13))

In [11]:
def make_user_id() -> str:
    """Random integer string, exactly 7 digits."""
    return str(random.randint(1_000_000, 9_999_999))

### START

In [12]:
@app.on_event("startup")
async def startup():
    init_db()

C:\Users\berna\AppData\Local\Temp\ipykernel_18456\4104724242.py:1: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


In [13]:
@app.get("/")
async def root():
    return {"message": "Flip Data Engineer API is running 🚀"}

In [14]:
@app.post("/ability")
async def get_ability(payload: AbilityRequest):
    raw_id = payload.raw_id or make_raw_id()
    user_id = payload.user_id or make_user_id()
    ability_id = payload.pokemon_ability_id
 
    # 1. Fetch from PokeAPI
    url = f"https://pokeapi.co/api/v2/ability/{ability_id}"
    async with httpx.AsyncClient(timeout=15) as client:
        resp = await client.get(url)
 
    if resp.status_code != 200:
        raise HTTPException(
            status_code=resp.status_code,
            detail=f"PokeAPI error for ability id {ability_id}",
        )
 
    data = resp.json()
 
    # 2. Extract effect_entries and pokemon list
    effect_entries = data.get("effect_entries", [])
    pokemon_list = [p["pokemon"]["name"] for p in data.get("pokemon", [])]
 
    if not effect_entries:
        raise HTTPException(status_code=404, detail="No effect_entries found.")
 
    # 3. Normalise and store in SQLite
    save_entries(
        raw_id=raw_id,
        user_id=user_id,
        pokemon_ability_id=int(ability_id),
        effect_entries=effect_entries,
    )
 
    # 4. Read back from DB and return
    stored = get_entries(
        raw_id=raw_id,
        user_id=user_id,
        pokemon_ability_id=int(ability_id),
    )
 
    return {
        "raw_id": raw_id,
        "user_id": user_id,
        "returned_entries": stored,
        "pokemon_list": pokemon_list,
    }

In [ ]:
uvicorn.run(app, host="0.0.0.0", port=8000)

INFO:     Started server process [18456]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:64869 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:64869 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:55640 - "POST /ability HTTP/1.1" 200 OK
